# Avellaneda-Stoikov Market Making Simulation

## KOSPI200 ATM Option LP Inventory Control Model

이 노트북은 Avellaneda-Stoikov(2008) 모델을 기반으로 KOSPI200 ATM 옵션 LP의 잔고관리 시뮬레이션을 구현합니다.

### 주요 특징
- **Reservation Price**: 순델타 노출 기반 내부 기준가격
- **Quote Control**: A-S 모델 기반 bid/ask 호가 배치
- **체결 시뮬레이션**: 포아송 프로세스 기반
- **데이터 소스**: 실제 틱 데이터 / Monte Carlo 전환 가능

### 데이터 설계
```
Mid Price = (매도호가1 + 매수호가1) / 2
Delta = 실제 데이터의 옵션델타
```

---
## 1. 환경 설정 및 Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Literal
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("Environment setup complete.")

---
## 2. 설정 및 파라미터

In [ ]:
@dataclass
class SimulationConfig:
    """
    시뮬레이션 설정 파라미터
    """
    # === 데이터 소스 설정 ===
    data_source: Literal['real', 'monte_carlo'] = 'real'  # 기본값: 실제 데이터
    real_data_path: str = '../data/kospi200_option_B0164812_20260316_1408.csv'
    
    # === 시뮬레이션 시간 설정 ===
    dt: float = 1.0                    # 시간 단위 (초)
    T_hours: float = 6.0               # 총 시뮬레이션 시간 (시간) - 장 운영시간 기준
    
    # === A-S 모델 파라미터 (논문 기본값) ===
    gamma: float = 0.1                 # 위험회피도
    sigma: float = 2.0                 # 변동성 (옵션 가격 기준)
    A: float = 140.0                   # 주문 유입 강도
    k: float = 1.5                     # 호가 민감도
    
    # === 델타 기반 확장 파라미터 ===
    phi: float = 0.05                  # 델타 inventory penalty 계수
    
    # === KRX 호가 단위 설정 ===
    tick_size: float = 0.05            # KRX KOSPI200 옵션 호가 단위
    apply_tick_size: bool = True       # 호가 단위 적용 여부
    
    # === Monte Carlo 설정 (monte_carlo 모드에서만 사용) ===
    mc_S0: float = 850.0               # 초기 지수 가격
    mc_M0: float = 40.0                # 초기 옵션 mid-price
    mc_delta0: float = 0.5             # 초기 ATM 델타
    mc_index_vol: float = 0.15         # 지수 연간 변동성
    
    # === 초기 상태 ===
    initial_cash: float = 0.0          # 초기 현금
    initial_inventory: int = 0         # 초기 옵션 재고
    
    # === 실제 데이터 모드 설정 ===
    use_data_length: bool = True       # True면 데이터 길이만큼만 시뮬레이션
    
    @property
    def T_seconds(self) -> float:
        """총 시뮬레이션 시간 (초)"""
        return self.T_hours * 3600
    
    @property
    def n_steps(self) -> int:
        """총 시뮬레이션 스텝 수"""
        return int(self.T_seconds / self.dt)


# 기본 설정 생성
config = SimulationConfig()
print(f"시뮬레이션 설정:")
print(f"  - 데이터 소스: {config.data_source}")
print(f"  - 데이터 경로: {config.real_data_path}")
print(f"  - 시간 단위: {config.dt}초")
print(f"  - A-S 파라미터: γ={config.gamma}, σ={config.sigma}, A={config.A}, k={config.k}")
print(f"  - 델타 penalty: φ={config.phi}")
print(f"  - KRX 호가 단위: {config.tick_size} (적용: {config.apply_tick_size})")

---
## 3. 데이터 로더 (Real / Monte Carlo Switching)

In [ ]:
@dataclass
class MarketSnapshot:
    """
    특정 시점의 시장 상태 스냅샷
    """
    timestamp: float           # 시뮬레이션 시간 (초)
    index_price: float         # KOSPI200 지수 가격
    option_mid: float          # 옵션 mid-price = (매도호가1 + 매수호가1) / 2
    option_delta: float        # 옵션 델타 (실제 데이터)
    best_bid: float            # 시장 best bid (매수호가1)
    best_ask: float            # 시장 best ask (매도호가1)
    bid_qty: int = 0           # best bid 잔량
    ask_qty: int = 0           # best ask 잔량
    theoretical_price: float = 0.0  # 이론가
    iv: float = 0.0            # 내재변동성


class DataLoader:
    """
    시장 데이터 로더 (Real / Monte Carlo 전환 가능)
    
    Real 모드:
        - option_mid = (매도호가1 + 매수호가1) / 2
        - option_delta = 옵션델타 컬럼
    """
    
    def __init__(self, config: SimulationConfig):
        self.config = config
        self.data_source = config.data_source
        
        if self.data_source == 'real':
            self._load_real_data()
        else:
            self._setup_monte_carlo()
    
    def _load_real_data(self):
        """실제 틱 데이터 로드"""
        print(f"데이터 로딩: {self.config.real_data_path}")
        self.df = pd.read_csv(self.config.real_data_path, encoding='utf-8-sig')
        
        # 시간 파싱
        self.df['timestamp'] = pd.to_datetime(self.df['시간'])
        
        # === 핵심: Mid Price 계산 ===
        # Mid = (매도호가1 + 매수호가1) / 2
        self.df['option_mid'] = (self.df['옵션매도호가1'] + self.df['옵션매수호가1']) / 2
        
        # 델타 (실제 데이터)
        self.df['option_delta'] = self.df['옵션델타']
        
        # 기타 컬럼 매핑
        self.df['index_price'] = self.df['지수현재가']
        self.df['best_bid'] = self.df['옵션매수호가1']
        self.df['best_ask'] = self.df['옵션매도호가1']
        self.df['bid_qty'] = self.df['옵션매수잔량1']
        self.df['ask_qty'] = self.df['옵션매도잔량1']
        self.df['theoretical_price'] = self.df['옵션이론가']
        self.df['iv'] = self.df['옵션IV']
        
        # NaN 처리 - 호가가 없는 행 제거
        required_cols = ['option_mid', 'option_delta', 'index_price', 'best_bid', 'best_ask']
        self.df = self.df.dropna(subset=required_cols)
        self.df = self.df.reset_index(drop=True)
        
        self.n_data_points = len(self.df)
        
        # 데이터 요약 출력
        print(f"\n데이터 로드 완료: {self.n_data_points} rows")
        print(f"\n데이터 요약:")
        print(f"  - 시간 범위: {self.df['timestamp'].iloc[0]} ~ {self.df['timestamp'].iloc[-1]}")
        print(f"  - 지수 범위: {self.df['index_price'].min():.2f} ~ {self.df['index_price'].max():.2f}")
        print(f"  - 옵션 Mid 범위: {self.df['option_mid'].min():.2f} ~ {self.df['option_mid'].max():.2f}")
        print(f"  - 델타 범위: {self.df['option_delta'].min():.4f} ~ {self.df['option_delta'].max():.4f}")
        print(f"  - 시장 스프레드 평균: {(self.df['best_ask'] - self.df['best_bid']).mean():.4f}")
    
    def _setup_monte_carlo(self):
        """Monte Carlo 시뮬레이션 설정"""
        self.rng = np.random.default_rng(seed=42)
        
        n_steps = self.config.n_steps
        dt_years = self.config.dt / (252 * 6 * 3600)
        
        # 지수 가격 경로 (GBM)
        dW = self.rng.standard_normal(n_steps) * np.sqrt(dt_years)
        log_returns = -0.5 * self.config.mc_index_vol**2 * dt_years + self.config.mc_index_vol * dW
        self.index_path = self.config.mc_S0 * np.exp(np.cumsum(log_returns))
        self.index_path = np.insert(self.index_path, 0, self.config.mc_S0)
        
        # 옵션 mid-price 경로
        option_noise = self.rng.standard_normal(n_steps + 1) * 0.1
        delta = self.config.mc_delta0
        self.option_mid_path = self.config.mc_M0 + delta * (self.index_path - self.config.mc_S0) + option_noise
        self.option_mid_path = np.maximum(self.option_mid_path, 0.01)
        
        # 델타 경로
        delta_noise = self.rng.standard_normal(n_steps + 1) * 0.02
        self.delta_path = np.clip(self.config.mc_delta0 + delta_noise, 0.3, 0.7)
        
        self.n_data_points = n_steps + 1
        print(f"Monte Carlo 시뮬레이션 설정 완료: {n_steps} steps")
    
    def get_snapshot(self, step: int) -> MarketSnapshot:
        """
        특정 스텝의 시장 스냅샷 반환
        """
        if self.data_source == 'real':
            # 실제 데이터: 범위 내에서 사용
            if step >= self.n_data_points:
                step = self.n_data_points - 1  # 마지막 데이터 재사용
            
            row = self.df.iloc[step]
            
            return MarketSnapshot(
                timestamp=step * self.config.dt,
                index_price=float(row['index_price']),
                option_mid=float(row['option_mid']),
                option_delta=float(row['option_delta']),
                best_bid=float(row['best_bid']),
                best_ask=float(row['best_ask']),
                bid_qty=int(row['bid_qty']) if pd.notna(row['bid_qty']) else 0,
                ask_qty=int(row['ask_qty']) if pd.notna(row['ask_qty']) else 0,
                theoretical_price=float(row['theoretical_price']) if pd.notna(row['theoretical_price']) else 0.0,
                iv=float(row['iv']) if pd.notna(row['iv']) else 0.0
            )
        else:
            # Monte Carlo
            option_mid = self.option_mid_path[step]
            spread = 0.25
            
            return MarketSnapshot(
                timestamp=step * self.config.dt,
                index_price=self.index_path[step],
                option_mid=option_mid,
                option_delta=self.delta_path[step],
                best_bid=option_mid - spread / 2,
                best_ask=option_mid + spread / 2,
                bid_qty=10,
                ask_qty=10
            )
    
    def get_n_steps(self) -> int:
        """사용 가능한 스텝 수 반환"""
        if self.data_source == 'real' and self.config.use_data_length:
            return self.n_data_points
        else:
            return self.config.n_steps


# 데이터 로더 테스트
loader = DataLoader(config)
snapshot = loader.get_snapshot(0)
print(f"\n첫 번째 스냅샷:")
print(f"  - 지수: {snapshot.index_price:.2f}")
print(f"  - 옵션 Mid: {snapshot.option_mid:.2f}")
print(f"  - 옵션 델타: {snapshot.option_delta:.4f}")
print(f"  - 시장 Bid/Ask: {snapshot.best_bid:.2f} / {snapshot.best_ask:.2f}")
print(f"  - 시장 스프레드: {snapshot.best_ask - snapshot.best_bid:.2f}")

---
## 4. A-S 모델 핵심 로직

In [ ]:
class AvellanedaStoikovModel:
    """
    Avellaneda-Stoikov Market Making 모델 (델타 기반 확장)
    
    핵심 수식:
    - Reservation Price: r = M - φ × q_Δ × (T-t)/T
    - Optimal Spread: δ_a + δ_b = γσ²(T-t) + (2/γ)ln(1 + γ/k)
    - Fill Intensity: λ(δ) = A × e^(-k×δ)
    
    KRX 호가 단위 적용:
    - Bid: 내림 (floor) → 더 낮은 가격에 매수 (보수적)
    - Ask: 올림 (ceil) → 더 높은 가격에 매도 (보수적)
    """
    
    def __init__(self, config: SimulationConfig):
        self.config = config
        self.gamma = config.gamma
        self.sigma = config.sigma
        self.A = config.A
        self.k = config.k
        self.phi = config.phi
        self.tick_size = config.tick_size
        self.apply_tick_size = config.apply_tick_size
    
    def _floor_to_tick(self, price: float) -> float:
        """가격을 tick size 단위로 내림"""
        return np.floor(price / self.tick_size) * self.tick_size
    
    def _ceil_to_tick(self, price: float) -> float:
        """가격을 tick size 단위로 올림"""
        return np.ceil(price / self.tick_size) * self.tick_size
    
    def _round_to_tick(self, price: float) -> float:
        """가격을 tick size 단위로 반올림"""
        return np.round(price / self.tick_size) * self.tick_size
    
    def compute_reservation_price(
        self,
        option_mid: float,
        net_delta: float,
        time_remaining_ratio: float  # (T-t)/T, 0~1 사이
    ) -> float:
        """
        델타 기반 Reservation Price 계산
        
        r = M - φ × q_Δ × (T-t)/T
        
        - net_delta > 0 (롱): r < M → 매도 유도
        - net_delta < 0 (숏): r > M → 매수 유도
        """
        reservation = option_mid - self.phi * net_delta * time_remaining_ratio
        return reservation
    
    def compute_optimal_spread(
        self,
        time_remaining_ratio: float
    ) -> float:
        """
        최적 스프레드 계산
        
        δ_a + δ_b = γσ²(T-t) + (2/γ)ln(1 + γ/k)
        """
        # 재고 리스크 보상 항
        inventory_term = self.gamma * (self.sigma ** 2) * time_remaining_ratio
        
        # 주문도착강도 보정 항
        intensity_term = (2 / self.gamma) * np.log(1 + self.gamma / self.k)
        
        total_spread = inventory_term + intensity_term
        
        return max(total_spread, 0.01)  # 최소 스프레드 보장
    
    def compute_quotes(
        self,
        option_mid: float,
        net_delta: float,
        time_remaining_ratio: float
    ) -> Tuple[float, float, float]:
        """
        최적 bid/ask 호가 계산
        
        Returns:
            (bid_price, ask_price, reservation_price)
        
        KRX 호가 단위 적용 시:
            - bid_price: 내림 (더 보수적인 매수)
            - ask_price: 올림 (더 보수적인 매도)
        """
        # Step 1: Reservation price
        r = self.compute_reservation_price(option_mid, net_delta, time_remaining_ratio)
        
        # Step 2: 최적 스프레드
        total_spread = self.compute_optimal_spread(time_remaining_ratio)
        half_spread = total_spread / 2
        
        # Step 3: Bid/Ask 배치 (연속적인 값)
        bid_price_raw = r - half_spread
        ask_price_raw = r + half_spread
        
        # Step 4: KRX 호가 단위 적용
        if self.apply_tick_size:
            bid_price = self._floor_to_tick(bid_price_raw)  # 내림 (보수적 매수)
            ask_price = self._ceil_to_tick(ask_price_raw)   # 올림 (보수적 매도)
        else:
            bid_price = bid_price_raw
            ask_price = ask_price_raw
        
        return bid_price, ask_price, r
    
    def compute_fill_probability(
        self,
        delta: float,
        dt: float,
        T_total: float
    ) -> float:
        """
        호가 체결 확률 계산 (포아송 프로세스)
        
        λ(δ) = A × e^(-k×δ)
        P(fill in dt) = λ(δ) × dt / T_total
        """
        delta = max(delta, 0)
        intensity = self.A * np.exp(-self.k * delta)
        prob = intensity * dt / T_total
        return min(prob, 1.0)


# 모델 테스트
model = AvellanedaStoikovModel(config)

test_mid = snapshot.option_mid
print(f"A-S 모델 테스트 (Mid = {test_mid:.2f}, Tick Size = {config.tick_size})")
print(f"{'='*60}")

# 케이스 1: 중립
bid, ask, r = model.compute_quotes(test_mid, net_delta=0.0, time_remaining_ratio=0.5)
print(f"\n[중립 포지션] net_delta = 0")
print(f"  Reservation (연속): {r:.4f}")
print(f"  LP Bid/Ask (틱 적용): {bid:.2f} / {ask:.2f}")
print(f"  Spread (틱 적용): {ask - bid:.2f}")

# 케이스 2: 롱 포지션
bid, ask, r = model.compute_quotes(test_mid, net_delta=2.0, time_remaining_ratio=0.5)
print(f"\n[롱 포지션] net_delta = +2.0")
print(f"  Reservation (연속): {r:.4f} (mid보다 {test_mid - r:.4f} 낮음 → 매도 유도)")
print(f"  LP Bid/Ask (틱 적용): {bid:.2f} / {ask:.2f}")

# 케이스 3: 숏 포지션
bid, ask, r = model.compute_quotes(test_mid, net_delta=-2.0, time_remaining_ratio=0.5)
print(f"\n[숏 포지션] net_delta = -2.0")
print(f"  Reservation (연속): {r:.4f} (mid보다 {r - test_mid:.4f} 높음 → 매수 유도)")
print(f"  LP Bid/Ask (틱 적용): {bid:.2f} / {ask:.2f}")

# 틱 적용 전후 비교
print(f"\n{'='*60}")
print(f"[틱 적용 전후 비교]")
model_no_tick = AvellanedaStoikovModel(SimulationConfig(apply_tick_size=False))
bid_raw, ask_raw, r = model_no_tick.compute_quotes(test_mid, net_delta=0.0, time_remaining_ratio=0.5)
bid_tick, ask_tick, r = model.compute_quotes(test_mid, net_delta=0.0, time_remaining_ratio=0.5)
print(f"  연속값:  Bid={bid_raw:.4f}, Ask={ask_raw:.4f}, Spread={ask_raw-bid_raw:.4f}")
print(f"  틱 적용: Bid={bid_tick:.2f}, Ask={ask_tick:.2f}, Spread={ask_tick-bid_tick:.2f}")

---
## 5. LP Agent 클래스

In [ ]:
@dataclass
class LPState:
    """
    LP의 현재 상태
    """
    cash: float = 0.0              # 현금 잔고
    inventory: int = 0             # 옵션 재고 (계약 수)
    net_delta: float = 0.0         # 순델타 노출 = inventory × delta
    
    # 호가 상태
    bid_price: float = 0.0
    ask_price: float = 0.0
    reservation_price: float = 0.0
    
    # 누적 통계
    total_bought: int = 0
    total_sold: int = 0
    total_trades: int = 0


class LPAgent:
    """
    LP (Liquidity Provider) Agent
    """
    
    def __init__(self, config: SimulationConfig, seed: int = 123):
        self.config = config
        self.model = AvellanedaStoikovModel(config)
        self.rng = np.random.default_rng(seed=seed)
        
        self.state = LPState(
            cash=config.initial_cash,
            inventory=config.initial_inventory
        )
        
        self.history: List[dict] = []
    
    def update_quotes(self, snapshot: MarketSnapshot, time_remaining_ratio: float):
        """
        시장 상태를 기반으로 호가 업데이트
        """
        # 순델타 계산: q_Δ = inventory × delta
        self.state.net_delta = self.state.inventory * snapshot.option_delta
        
        # A-S 모델로 최적 호가 계산
        bid, ask, r = self.model.compute_quotes(
            option_mid=snapshot.option_mid,
            net_delta=self.state.net_delta,
            time_remaining_ratio=time_remaining_ratio
        )
        
        self.state.bid_price = bid
        self.state.ask_price = ask
        self.state.reservation_price = r
    
    def simulate_fills(self, snapshot: MarketSnapshot, dt: float, T_total: float) -> Tuple[bool, bool]:
        """
        포아송 프로세스 기반 체결 시뮬레이션
        
        δ_bid = option_mid - LP_bid (LP bid가 mid보다 얼마나 낮은지)
        δ_ask = LP_ask - option_mid (LP ask가 mid보다 얼마나 높은지)
        """
        # Bid 체결 확률
        delta_bid = snapshot.option_mid - self.state.bid_price
        prob_bid = self.model.compute_fill_probability(delta_bid, dt, T_total)
        
        # Ask 체결 확률
        delta_ask = self.state.ask_price - snapshot.option_mid
        prob_ask = self.model.compute_fill_probability(delta_ask, dt, T_total)
        
        # 체결 판정
        bid_filled = self.rng.random() < prob_bid
        ask_filled = self.rng.random() < prob_ask
        
        return bid_filled, ask_filled
    
    def execute_fills(self, bid_filled: bool, ask_filled: bool):
        """
        체결 실행
        """
        if bid_filled:
            self.state.inventory += 1
            self.state.cash -= self.state.bid_price
            self.state.total_bought += 1
            self.state.total_trades += 1
        
        if ask_filled:
            self.state.inventory -= 1
            self.state.cash += self.state.ask_price
            self.state.total_sold += 1
            self.state.total_trades += 1
    
    def compute_pnl(self, current_mid: float) -> float:
        """현재 P&L (Mark-to-Market)"""
        return self.state.cash + self.state.inventory * current_mid
    
    def record_state(self, step: int, snapshot: MarketSnapshot, time_remaining_ratio: float):
        """상태 기록"""
        self.history.append({
            'step': step,
            'timestamp': snapshot.timestamp,
            'index_price': snapshot.index_price,
            'option_mid': snapshot.option_mid,
            'option_delta': snapshot.option_delta,
            'market_bid': snapshot.best_bid,
            'market_ask': snapshot.best_ask,
            'market_spread': snapshot.best_ask - snapshot.best_bid,
            'cash': self.state.cash,
            'inventory': self.state.inventory,
            'net_delta': self.state.net_delta,
            'reservation_price': self.state.reservation_price,
            'lp_bid': self.state.bid_price,
            'lp_ask': self.state.ask_price,
            'lp_spread': self.state.ask_price - self.state.bid_price,
            'pnl': self.compute_pnl(snapshot.option_mid),
            'time_remaining_ratio': time_remaining_ratio,
            'total_trades': self.state.total_trades,
            'total_bought': self.state.total_bought,
            'total_sold': self.state.total_sold
        })
    
    def get_history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.history)


print("LPAgent 클래스 정의 완료")

---
## 6. 시뮬레이션 엔진

In [ ]:
class Simulator:
    """
    A-S Market Making 시뮬레이션 엔진
    """
    
    def __init__(self, config: SimulationConfig, seed: int = 123):
        self.config = config
        self.loader = DataLoader(config)
        self.agent = LPAgent(config, seed=seed)
    
    def run(self, verbose: bool = True) -> pd.DataFrame:
        """
        시뮬레이션 실행
        """
        n_steps = self.loader.get_n_steps()
        dt = self.config.dt
        
        # T_total: 실제 데이터면 데이터 길이, 아니면 설정값
        T_total = n_steps * dt
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"시뮬레이션 시작")
            print(f"  - 총 스텝: {n_steps}")
            print(f"  - 시간 단위: {dt}초")
            print(f"  - 총 시간: {T_total:.0f}초 ({T_total/60:.1f}분)")
            print(f"{'='*60}")
        
        for step in range(n_steps):
            # 현재 시간 및 남은 시간 비율
            current_time = step * dt
            time_remaining_ratio = (T_total - current_time) / T_total
            
            # 시장 스냅샷
            snapshot = self.loader.get_snapshot(step)
            
            # 호가 업데이트
            self.agent.update_quotes(snapshot, time_remaining_ratio)
            
            # 체결 시뮬레이션
            bid_filled, ask_filled = self.agent.simulate_fills(snapshot, dt, T_total)
            
            # 체결 실행
            self.agent.execute_fills(bid_filled, ask_filled)
            
            # 상태 기록
            self.agent.record_state(step, snapshot, time_remaining_ratio)
            
            # 진행 상황 출력
            if verbose and n_steps >= 10 and (step + 1) % (n_steps // 10) == 0:
                pct = (step + 1) / n_steps * 100
                pnl = self.agent.compute_pnl(snapshot.option_mid)
                print(f"  [{pct:5.1f}%] Step {step+1:5d} | "
                      f"Inv: {self.agent.state.inventory:+3d} | "
                      f"NetΔ: {self.agent.state.net_delta:+6.2f} | "
                      f"P&L: {pnl:+8.2f} | "
                      f"Trades: {self.agent.state.total_trades}")
        
        if verbose:
            print(f"{'='*60}")
            print(f"시뮬레이션 완료!")
        
        return self.agent.get_history_df()


print("Simulator 클래스 정의 완료")

---
## 7. 시뮬레이션 실행 (실제 데이터)

In [ ]:
# 실제 데이터 기반 시뮬레이션 실행
config = SimulationConfig(
    data_source='real',
    real_data_path='../data/kospi200_option_B0164812_20260316_1408.csv',
    use_data_length=True,  # 데이터 길이만큼만 시뮬레이션
    gamma=0.1,
    sigma=2.0,
    A=140.0,
    k=1.5,
    phi=0.05
)

simulator = Simulator(config, seed=42)
results = simulator.run(verbose=True)

In [ ]:
# 결과 요약
print("\n" + "="*60)
print("시뮬레이션 결과 요약")
print("="*60)

final = results.iloc[-1]
print(f"\n[최종 상태]")
print(f"  - P&L: {final['pnl']:.2f}")
print(f"  - 재고: {final['inventory']} 계약")
print(f"  - 순델타: {final['net_delta']:.4f}")
print(f"  - 현금: {final['cash']:.2f}")

print(f"\n[거래 통계]")
print(f"  - 총 체결: {final['total_trades']}회")
print(f"  - 매수: {final['total_bought']}회")
print(f"  - 매도: {final['total_sold']}회")

print(f"\n[P&L 통계]")
print(f"  - 평균: {results['pnl'].mean():.2f}")
print(f"  - 표준편차: {results['pnl'].std():.2f}")
print(f"  - 최대: {results['pnl'].max():.2f}")
print(f"  - 최소: {results['pnl'].min():.2f}")

print(f"\n[재고 통계]")
print(f"  - 평균: {results['inventory'].mean():.2f}")
print(f"  - 표준편차: {results['inventory'].std():.2f}")
print(f"  - 최대: {results['inventory'].max()}")
print(f"  - 최소: {results['inventory'].min()}")

print(f"\n[호가 통계]")
print(f"  - LP 스프레드 평균: {results['lp_spread'].mean():.4f}")
print(f"  - 시장 스프레드 평균: {results['market_spread'].mean():.4f}")

---
## 8. 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# X축: 스텝 번호
x = results['step']

# 1. P&L 추이
ax1 = axes[0, 0]
ax1.plot(x, results['pnl'], 'b-', linewidth=0.8)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.fill_between(x, 0, results['pnl'], alpha=0.3, color='blue')
ax1.set_xlabel('Step')
ax1.set_ylabel('P&L')
ax1.set_title('P&L 추이')
ax1.grid(True, alpha=0.3)

# 2. 재고 추이
ax2 = axes[0, 1]
ax2.plot(x, results['inventory'], 'g-', linewidth=0.8)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.fill_between(x, 0, results['inventory'], alpha=0.3, color='green')
ax2.set_xlabel('Step')
ax2.set_ylabel('Inventory (계약 수)')
ax2.set_title('옵션 재고 추이')
ax2.grid(True, alpha=0.3)

# 3. 순델타 노출
ax3 = axes[1, 0]
ax3.plot(x, results['net_delta'], 'r-', linewidth=0.8)
ax3.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax3.fill_between(x, 0, results['net_delta'], alpha=0.3, color='red')
ax3.set_xlabel('Step')
ax3.set_ylabel('Net Delta')
ax3.set_title('순델타 노출 추이')
ax3.grid(True, alpha=0.3)

# 4. 가격 및 LP 호가
ax4 = axes[1, 1]
ax4.plot(x, results['option_mid'], 'k-', label='Mid Price', linewidth=1)
ax4.plot(x, results['reservation_price'], 'b--', label='Reservation', linewidth=0.8, alpha=0.7)
ax4.fill_between(x, results['lp_bid'], results['lp_ask'], 
                 alpha=0.2, color='green', label='LP Spread')
ax4.set_xlabel('Step')
ax4.set_ylabel('Price')
ax4.set_title('옵션 가격 및 LP 호가')
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('simulation_results_real_data.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n그래프가 'simulation_results_real_data.png'로 저장되었습니다.")

In [ ]:
# LP 호가 vs 시장 호가 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. LP 호가 vs 시장 호가 (일부 구간 확대)
ax1 = axes[0]
sample_range = slice(0, min(200, len(results)))  # 처음 200개 스텝
x_sample = results['step'].iloc[sample_range]

ax1.plot(x_sample, results['market_ask'].iloc[sample_range], 'r-', label='Market Ask', alpha=0.7)
ax1.plot(x_sample, results['lp_ask'].iloc[sample_range], 'r--', label='LP Ask', alpha=0.7)
ax1.plot(x_sample, results['option_mid'].iloc[sample_range], 'k-', label='Mid', linewidth=1.5)
ax1.plot(x_sample, results['lp_bid'].iloc[sample_range], 'b--', label='LP Bid', alpha=0.7)
ax1.plot(x_sample, results['market_bid'].iloc[sample_range], 'b-', label='Market Bid', alpha=0.7)

ax1.set_xlabel('Step')
ax1.set_ylabel('Price')
ax1.set_title('LP 호가 vs 시장 호가 (처음 200 스텝)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# 2. 스프레드 비교
ax2 = axes[1]
ax2.plot(x, results['market_spread'], 'b-', label='Market Spread', alpha=0.7, linewidth=0.8)
ax2.plot(x, results['lp_spread'], 'r-', label='LP Spread', alpha=0.7, linewidth=0.8)
ax2.axhline(y=results['market_spread'].mean(), color='b', linestyle='--', alpha=0.5, 
            label=f'Market Avg: {results["market_spread"].mean():.3f}')
ax2.axhline(y=results['lp_spread'].mean(), color='r', linestyle='--', alpha=0.5,
            label=f'LP Avg: {results["lp_spread"].mean():.3f}')
ax2.set_xlabel('Step')
ax2.set_ylabel('Spread')
ax2.set_title('스프레드 비교')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spread_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 여러 번 시뮬레이션 (다른 시드)

In [ ]:
# 동일 데이터에서 체결의 랜덤성만 바꿔가며 여러 번 시뮬레이션
n_simulations = 50
final_pnls = []
final_inventories = []
final_trades = []

print(f"{n_simulations}회 시뮬레이션 실행 중...")

for i in range(n_simulations):
    sim = Simulator(config, seed=i)
    res = sim.run(verbose=False)
    
    final_pnls.append(res['pnl'].iloc[-1])
    final_inventories.append(res['inventory'].iloc[-1])
    final_trades.append(res['total_trades'].iloc[-1])
    
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{n_simulations} 완료")

final_pnls = np.array(final_pnls)
final_inventories = np.array(final_inventories)
final_trades = np.array(final_trades)

print(f"\n{n_simulations}회 시뮬레이션 결과:")
print(f"  P&L: 평균 {final_pnls.mean():.2f}, 표준편차 {final_pnls.std():.2f}")
print(f"  재고: 평균 {final_inventories.mean():.2f}, 표준편차 {final_inventories.std():.2f}")
print(f"  체결: 평균 {final_trades.mean():.1f}, 표준편차 {final_trades.std():.1f}")

In [ ]:
# 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# P&L 분포
ax1 = axes[0]
ax1.hist(final_pnls, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(x=final_pnls.mean(), color='red', linestyle='--', 
            label=f'평균: {final_pnls.mean():.2f}')
ax1.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax1.set_xlabel('최종 P&L')
ax1.set_ylabel('빈도')
ax1.set_title(f'P&L 분포 ({n_simulations}회)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 재고 분포
ax2 = axes[1]
ax2.hist(final_inventories, bins=20, edgecolor='black', alpha=0.7, color='forestgreen')
ax2.axvline(x=final_inventories.mean(), color='red', linestyle='--',
            label=f'평균: {final_inventories.mean():.2f}')
ax2.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('최종 재고')
ax2.set_ylabel('빈도')
ax2.set_title(f'최종 재고 분포 ({n_simulations}회)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 체결 횟수 분포
ax3 = axes[2]
ax3.hist(final_trades, bins=20, edgecolor='black', alpha=0.7, color='coral')
ax3.axvline(x=final_trades.mean(), color='red', linestyle='--',
            label=f'평균: {final_trades.mean():.1f}')
ax3.set_xlabel('총 체결 횟수')
ax3.set_ylabel('빈도')
ax3.set_title(f'체결 횟수 분포 ({n_simulations}회)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('simulation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. 파라미터 민감도 분석

In [ ]:
# γ (위험회피도) 변화에 따른 결과 비교
gammas = [0.01, 0.05, 0.1, 0.5, 1.0]
gamma_results = {}

print("γ (위험회피도) 민감도 분석...")
print(f"{'='*70}")

for gamma in gammas:
    config_test = SimulationConfig(
        data_source='real',
        real_data_path='../data/kospi200_option_B0164812_20260316_1408.csv',
        use_data_length=True,
        gamma=gamma,
        sigma=2.0,
        A=140.0,
        k=1.5,
        phi=0.05
    )
    
    sim = Simulator(config_test, seed=42)
    res = sim.run(verbose=False)
    
    gamma_results[gamma] = {
        'pnl_final': res['pnl'].iloc[-1],
        'pnl_std': res['pnl'].std(),
        'inventory_std': res['inventory'].std(),
        'inventory_final': res['inventory'].iloc[-1],
        'trades': res['total_trades'].iloc[-1],
        'lp_spread_avg': res['lp_spread'].mean()
    }
    
    print(f"  γ={gamma:5.2f}: P&L={res['pnl'].iloc[-1]:+7.2f}, "
          f"Inv.Std={res['inventory'].std():5.2f}, "
          f"Spread={res['lp_spread'].mean():.4f}, "
          f"Trades={res['total_trades'].iloc[-1]}")

# 결과 테이블
print(f"\n{'='*70}")
gamma_df = pd.DataFrame(gamma_results).T
gamma_df.index.name = 'gamma'
print("\n결과 요약:")
print(gamma_df.round(3))

In [ ]:
# 민감도 분석 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gammas_list = list(gamma_results.keys())

# 1. P&L vs γ
ax1 = axes[0]
pnls = [gamma_results[g]['pnl_final'] for g in gammas_list]
ax1.plot(gammas_list, pnls, 'bo-', markersize=8)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('γ (위험회피도)')
ax1.set_ylabel('최종 P&L')
ax1.set_title('γ vs P&L')
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

# 2. 재고 표준편차 vs γ
ax2 = axes[1]
inv_stds = [gamma_results[g]['inventory_std'] for g in gammas_list]
ax2.plot(gammas_list, inv_stds, 'go-', markersize=8)
ax2.set_xlabel('γ (위험회피도)')
ax2.set_ylabel('재고 표준편차')
ax2.set_title('γ vs 재고 변동성')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)

# 3. LP 스프레드 vs γ
ax3 = axes[2]
spreads = [gamma_results[g]['lp_spread_avg'] for g in gammas_list]
ax3.plot(gammas_list, spreads, 'ro-', markersize=8)
ax3.set_xlabel('γ (위험회피도)')
ax3.set_ylabel('평균 LP 스프레드')
ax3.set_title('γ vs LP 스프레드')
ax3.set_xscale('log')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gamma_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n분석 결과:")
print("  - γ 증가 → LP 스프레드 증가 (더 보수적)")
print("  - γ 증가 → 재고 변동성 감소 (inventory control 강화)")
print("  - γ 증가 → 체결 횟수 감소 (호가가 멀어짐)")

---
## 11. 결과 데이터 저장

In [ ]:
# 상세 결과 CSV 저장
results.to_csv('simulation_results_detail.csv', index=False, encoding='utf-8-sig')
print("상세 결과가 'simulation_results_detail.csv'로 저장되었습니다.")

# 결과 미리보기
print("\n결과 데이터 미리보기:")
results.head(10)